# Datasets processing

In [1]:
# Processing procedures of the data we are using 

# Adult Income     https://archive.ics.uci.edu/ml/datasets/adult
# COMPAS           https://www.kaggle.com/datasets/danofer/compass
# Diabetes         https://archive.ics.uci.edu/dataset/296/diabetes+130-us+hospitals+for+years+1999-2008
# German Credit    https://archive.ics.uci.edu/dataset/144/statlog+german+credit+data
# HELOC            https://community.fico.com/s/explainable-machine-learning-challenge
# HIGGS            OpenML
# Independent      
# Synthetic        OpenXAI

In [2]:
import time
import numpy as np
import pandas as pd
import zipfile as zf
import nbimporter

import sys

import sklearn.ensemble
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [3]:
import Taylor_Explainer as texp

# Data preprocessing methods

In [21]:
# each categorical attribute is converted in 0/1 attributes using pandas.get_dummies
# drop_first=True only for attributes with binary values

# receive data as a pandas.DataFrame with categorical attributes
# cat_cols list indicating the columns names from df are numerical

# RETURN one-hot encoded data

def get_dummies_drop_first_only_binary_atts(data, cat_cols):
    df= data.copy()

    for i in cat_cols:
        if len(df.groupby([i]).size())> 2:
            df= pd.get_dummies(df, prefix=[i], columns=[i])
        else:
            df= pd.get_dummies(df, prefix=[i], columns=[i], drop_first=True)
    
    return df

# Adult Income

# COMPAS

# Diabetes

# German Credit

In [4]:
raw_data_ger= pd.read_csv('data/german_credit_data.csv', index_col='ID')

raw_data_ger.head()

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
ID,,,,,,,,,
0,67,male,2,own,NaN,little,1169,6,radio/TV
1,22,female,2,own,little,moderate,5951,48,radio/TV
2,49,male,1,own,little,NaN,2096,12,education
3,45,male,2,free,little,little,7882,42,furniture/equipment
4,53,male,2,free,little,little,4870,24,car


In [5]:
# get the target column (y)

raw_data_ger_target= pd.read_csv('data/german.data', delimiter=' ', header=None)

t_pos= raw_data_ger_target.shape[1]

y_ger= raw_data_ger_target.loc[:,raw_data_ger_target.columns[(t_pos-1):t_pos]]
y_ger= y_ger.replace(2, 0)

y_ger.head()

,20
0,1
1,0
2,1
3,1
4,0


In [6]:
categor_columns_ger= ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']
numeric_columns_ger= list(filter(lambda x:x not in categor_columns_ger, raw_data_ger.columns))

numeric_columns_ger

['Age', 'Job', 'Credit amount', 'Duration']

In [7]:
#Check for null values

texp.check_null_values(raw_data_ger, text_info=True)

There are some missing values in the dataset


In [8]:
# fill missing data
# numerical features are filled with mean values
x_ger= texp.pre_proc_fillna_num_fts(raw_data_ger, numeric_columns_ger, num_type='mean')
# categorical features are filled with mode values
x_ger= texp.pre_proc_fillna_cat_fts(x_ger, categor_columns_ger, cat_type='mode')

In [23]:
# one-hot encoding the categorical features

x_ger_ohe= pd.get_dummies(x_ger, columns=categor_columns_ger)

In [24]:
categor_columns_ger_ohe= list(filter(lambda x:x not in numeric_columns_ger, x_ger_ohe.columns))

categor_columns_ger_ohe

['Sex_female',
 'Sex_male',
 'Housing_free',
 'Housing_own',
 'Housing_rent',
 'Saving accounts_little',
 'Saving accounts_moderate',
 'Saving accounts_quite rich',
 'Saving accounts_rich',
 'Checking account_little',
 'Checking account_moderate',
 'Checking account_rich',
 'Purpose_business',
 'Purpose_car',
 'Purpose_domestic appliances',
 'Purpose_education',
 'Purpose_furniture/equipment',
 'Purpose_radio/TV',
 'Purpose_repairs',
 'Purpose_vacation/others']

In [25]:
# one-hot encoding the categorical features with drop_first only for binary attributes

x_ger_ohe= get_dummies_drop_first_only_binary_atts(x_ger, categor_columns_ger)

In [26]:
categor_columns_ger_ohe= list(filter(lambda x:x not in numeric_columns_ger, x_ger_ohe.columns))

categor_columns_ger_ohe

['Sex_male',
 'Housing_free',
 'Housing_own',
 'Housing_rent',
 'Saving accounts_little',
 'Saving accounts_moderate',
 'Saving accounts_quite rich',
 'Saving accounts_rich',
 'Checking account_little',
 'Checking account_moderate',
 'Checking account_rich',
 'Purpose_business',
 'Purpose_car',
 'Purpose_domestic appliances',
 'Purpose_education',
 'Purpose_furniture/equipment',
 'Purpose_radio/TV',
 'Purpose_repairs',
 'Purpose_vacation/others']

In [27]:
x_ger_ohe.shape

# original data --  9 features
# ohe data      -- 23 features 

# numeric fts   --  4 features
# categorical   --  5 features
# ohe fts       -- 19 features

(1000, 23)

In [28]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_ger_ohe= texp.normalize_selected(x_ger_ohe)

In [29]:
train_ger, test_ger, labels_train_ger, labels_test_ger= sklearn.model_selection.train_test_split(x_ger_ohe,
                                                                                             y_ger,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_ger.head()

,Age,Job,Credit amount,Duration,Sex_male,Housing_free,Housing_own,Housing_rent,Saving accounts_little,Saving accounts_moderate,...,Checking account_moderate,Checking account_rich,Purpose_business,Purpose_car,Purpose_domestic appliances,Purpose_education,Purpose_furniture/equipment,Purpose_radio/TV,Purpose_repairs,Purpose_vacation/others
ID,,,,,,,,,,,,,,,,,,,,,
281,0.553571,0.666667,0.072851,0.117647,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
42,0.446429,0.333333,0.327611,0.205882,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
255,0.142857,0.333333,0.394410,0.823529,1.0,0.0,1.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
906,0.089286,0.333333,0.193298,0.250000,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
394,0.214286,1.000000,0.118631,0.073529,1.0,0.0,1.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# HELOC

In [39]:
heloc= pd.read_csv('data/heloc_dataset_cleaned.csv')

# split synth into features (x) and target (y)
x_heloc= heloc.loc[:,heloc.columns[1:24]]
y_heloc= heloc.loc[:,heloc.columns[0:1]]

x_heloc.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,55.0,144.0,4,84,20,3,0,83,2.0,3,...,43,0.0,0,0,33.0,?,8.0,1.0,1.0,69.0
1,61.0,58.0,15,41,2,4,4,100,?,0,...,67,0.0,0,0,0.0,?,0.0,?,?,0.0
2,67.0,66.0,5,24,9,0,0,100,?,7,...,44,0.0,4,4,53.0,66.0,4.0,2.0,1.0,86.0
3,66.0,169.0,1,73,28,1,1,93,76.0,6,...,57,0.0,5,4,72.0,83.0,6.0,4.0,3.0,91.0
4,81.0,333.0,27,132,12,0,0,100,?,7,...,25,0.0,1,1,51.0,89.0,3.0,1.0,0.0,80.0


In [40]:
print(np.round(x_heloc.memory_usage().sum() / 10**6, 2), "MB")

1.82 MB


In [41]:
y_heloc.head()

,RiskPerformance
0,Bad
1,Bad
2,Bad
3,Bad
4,Bad


In [42]:
y_heloc= y_heloc.replace(to_replace=['Bad', 'Good'], value=[0, 1])

y_heloc.head()

,RiskPerformance
0,0
1,0
2,0
3,0
4,0


In [43]:
# there are missing values filled with a '?' character. we replaced it with nan values
x_heloc_clean= x_heloc.replace('?', np.nan)
x_heloc_clean.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,MaxDelq2PublicRecLast12M,...,PercentInstallTrades,MSinceMostRecentInqexcl7days,NumInqLast6M,NumInqLast6Mexcl7days,NetFractionRevolvingBurden,NetFractionInstallBurden,NumRevolvingTradesWBalance,NumInstallTradesWBalance,NumBank2NatlTradesWHighUtilization,PercentTradesWBalance
0,55.0,144.0,4,84,20,3,0,83,2.0,3,...,43,0.0,0,0,33.0,NaN,8.0,1.0,1.0,69.0
1,61.0,58.0,15,41,2,4,4,100,NaN,0,...,67,0.0,0,0,0.0,NaN,0.0,NaN,NaN,0.0
2,67.0,66.0,5,24,9,0,0,100,NaN,7,...,44,0.0,4,4,53.0,66.0,4.0,2.0,1.0,86.0
3,66.0,169.0,1,73,28,1,1,93,76.0,6,...,57,0.0,5,4,72.0,83.0,6.0,4.0,3.0,91.0
4,81.0,333.0,27,132,12,0,0,100,NaN,7,...,25,0.0,1,1,51.0,89.0,3.0,1.0,0.0,80.0


In [44]:
#Check for null values

texp.check_null_values(x_heloc_clean, text_info=True)

There are some missing values in the dataset


In [45]:
categor_columns_heloc= ['MaxDelq2PublicRecLast12M','MaxDelqEver']
numeric_columns_heloc= list(filter(lambda x:x not in categor_columns_heloc, x_heloc_clean.columns))

In [46]:
# fill missing data
# numerical features are filled with mean values
x_heloc_clean= texp.pre_proc_fillna_num_fts((x_heloc_clean.astype(float)), numeric_columns_heloc, 
                                            num_type='mean')
# categorical features are filled with mode values
x_heloc_clean= texp.pre_proc_fillna_cat_fts(x_heloc_clean, categor_columns_heloc, cat_type='mode')

In [51]:
# one-hot encoding the categorical features

x_heloc_ohe= pd.get_dummies(x_heloc_clean, columns=categor_columns_heloc)

In [48]:
categor_columns_heloc_ohe= list(filter(lambda x:x not in numeric_columns_heloc, x_heloc_ohe.columns))

categor_columns_heloc_ohe

['MaxDelq2PublicRecLast12M_0.0',
 'MaxDelq2PublicRecLast12M_1.0',
 'MaxDelq2PublicRecLast12M_2.0',
 'MaxDelq2PublicRecLast12M_3.0',
 'MaxDelq2PublicRecLast12M_4.0',
 'MaxDelq2PublicRecLast12M_5.0',
 'MaxDelq2PublicRecLast12M_6.0',
 'MaxDelq2PublicRecLast12M_7.0',
 'MaxDelq2PublicRecLast12M_9.0',
 'MaxDelqEver_2.0',
 'MaxDelqEver_3.0',
 'MaxDelqEver_4.0',
 'MaxDelqEver_5.0',
 'MaxDelqEver_6.0',
 'MaxDelqEver_7.0',
 'MaxDelqEver_8.0']

In [53]:
x_heloc_ohe.shape

# original data -- 23 features
# ohe data      -- 37 features 

# numeric fts   -- 21 features
# categorical   --  2 features
# ohe fts       -- 16 features

(9871, 37)

In [49]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_heloc_ohe= texp.normalize_selected(x_heloc_ohe)

In [50]:
train_hel, test_hel, labels_train_hel, labels_test_hel= sklearn.model_selection.train_test_split(x_heloc_ohe,
                                                                                             y_heloc,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_hel.head()

,ExternalRiskEstimate,MSinceOldestTradeOpen,MSinceMostRecentTradeOpen,AverageMInFile,NumSatisfactoryTrades,NumTrades60Ever2DerogPubRec,NumTrades90Ever2DerogPubRec,PercentTradesNeverDelq,MSinceMostRecentDelq,NumTotalTrades,...,MaxDelq2PublicRecLast12M_6.0,MaxDelq2PublicRecLast12M_7.0,MaxDelq2PublicRecLast12M_9.0,MaxDelqEver_2.0,MaxDelqEver_3.0,MaxDelqEver_4.0,MaxDelqEver_5.0,MaxDelqEver_6.0,MaxDelqEver_7.0,MaxDelqEver_8.0
720,0.573770,0.072409,0.018277,0.071240,0.139241,0.052632,0.0,0.92,0.325301,0.115385,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1759,0.442623,0.151061,0.002611,0.113456,0.227848,0.000000,0.0,1.00,0.263609,0.192308,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3964,0.524590,0.067416,0.007833,0.044855,0.088608,0.000000,0.0,0.86,0.457831,0.067308,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1786,0.885246,0.159800,0.031332,0.110818,0.329114,0.000000,0.0,1.00,0.263609,0.250000,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
9428,0.377049,0.238452,0.120104,0.221636,0.278481,0.000000,0.0,0.78,0.036145,0.221154,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


# HIGGS

In [4]:
ds= zf.ZipFile('data/higgs_98k.zip')

higgs= pd.read_csv(ds.open('higgs_98k.csv'), low_memory=False)

# split synth into features (x) and target (y)
x_higgs= higgs.loc[:,higgs.columns[1:29]]
y_higgs= higgs.loc[:,higgs.columns[0:1]]

x_higgs.head()

,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet1pt,jet1eta,jet1phi,jet1b-tag,jet2pt,...,jet4eta,jet4phi,jet4b-tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
0,0.907542,0.329147,0.359412,1.497970,-0.313010,1.095531,-0.557525,-1.588230,2.173076,0.812581,...,-1.138930,-0.000819110195152462,0,0.302219897508621,0.833048164844513,0.985699653625488,0.978098392486572,0.779732167720795,0.992355763912201,0.79834258556366
1,0.798835,1.470639,-1.635975,0.453773,0.425629,1.104875,1.282322,1.381664,0.000000,0.851737,...,1.128848,0.900460839271545,0,0.909753262996674,1.10833048820496,0.985692203044891,0.951331257820129,0.803251504898071,0.865924417972565,0.780117571353912
2,1.344385,-0.876626,0.935913,1.992050,0.882454,1.786066,-1.646778,-0.942383,0.000000,2.423265,...,-0.678379,-1.36035633087158,0,0.946652472019196,1.0287036895752,0.998656094074249,0.728280603885651,0.869200229644775,1.02673649787903,0.957903981208801
3,1.105009,0.321356,1.522401,0.882808,-1.205349,0.681466,-1.070464,-0.921871,0.000000,0.800872,...,-0.373566,0.113040611147881,0,0.755856454372406,1.36105704307556,0.986609697341919,0.838084638118744,1.13329517841339,0.872244894504547,0.808486521244049
4,1.595839,-0.607811,0.007075,1.818450,-0.111906,0.847550,-0.566437,1.581239,2.173076,0.755421,...,-0.654227,-1.27434492111206,3.10196137428284,0.823760569095612,0.938191413879395,0.971758186817169,0.789176344871521,0.430553287267685,0.961356937885284,0.957817912101746


In [14]:
print(np.round(x_higgs.memory_usage().sum() / 10**6, 2), "MB")

21.96 MB


In [15]:
y_higgs.head()

,class
0,1
1,1
2,0
3,1
4,0


In [17]:
# Check for null values

texp.check_null_values(x_higgs, text_info=True)

# All the values are numeric in the HIGGS dataset

There are no missing values in the dataset


In [18]:
x_higgs.isin(['?']).sum(axis=0)

lepton_pT                   0
lepton_eta                  0
lepton_phi                  0
missing_energy_magnitude    0
missing_energy_phi          0
jet1pt                      0
jet1eta                     0
jet1phi                     0
jet1b-tag                   0
jet2pt                      0
jet2eta                     0
jet2phi                     0
jet2b-tag                   0
jet3pt                      0
jet3eta                     0
jet3phi                     0
jet3b-tag                   0
jet4pt                      0
jet4eta                     0
jet4phi                     1
jet4b-tag                   1
m_jj                        1
m_jjj                       1
m_lv                        1
m_jlv                       1
m_bb                        1
m_wbb                       1
m_wwbb                      1
dtype: int64

In [19]:
x_higgs= x_higgs.replace('?', np.nan)

x_higgs= x_higgs.astype(float)

# fill missing (numeric) data with mean values
x_higgs= texp.pre_proc_fillna_num_fts(x_higgs, x_higgs.columns, num_type='mean')

In [20]:
# normalize numeric columns of the dataframe with values between 0 and 1

x_higgs= texp.normalize_selected(x_higgs)

In [21]:
train_hig, test_hig, labels_train_hig, labels_test_hig= sklearn.model_selection.train_test_split(x_higgs,
                                                                                             y_higgs,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)

train_hig.head()

,lepton_pT,lepton_eta,lepton_phi,missing_energy_magnitude,missing_energy_phi,jet1pt,jet1eta,jet1phi,jet1b-tag,jet2pt,...,jet4eta,jet4phi,jet4b-tag,m_jj,m_jjj,m_lv,m_jlv,m_bb,m_wbb,m_wwbb
40951,0.022891,0.759352,0.876317,0.125775,0.364631,0.100052,0.691679,0.540606,1.0,0.027402,...,0.758086,0.287798,0.0,0.018702,0.056143,0.192729,0.067751,0.139907,0.112800,0.086516
76614,0.073727,0.395679,0.671134,0.159534,0.417735,0.111667,0.538436,0.925663,0.0,0.082393,...,0.108536,0.276655,0.0,0.091208,0.082455,0.199951,0.073763,0.085361,0.096756,0.100368
97143,0.023231,0.716143,0.401294,0.127702,0.715179,0.055456,0.571619,0.800706,1.0,0.085941,...,0.727076,0.213142,0.0,0.030815,0.035638,0.193884,0.105637,0.077562,0.076504,0.070331
57634,0.424256,0.546509,0.804049,0.080008,0.627251,0.166157,0.640654,0.175576,0.0,0.244317,...,0.591030,0.476586,0.0,0.044032,0.087570,0.215782,0.060640,0.161801,0.162391,0.176001
16835,0.201448,0.663533,0.685778,0.074547,0.956967,0.172640,0.591796,0.181306,1.0,0.015514,...,0.236746,0.556842,0.0,0.035411,0.110135,0.213415,0.061279,0.064803,0.089685,0.091670


# Independent

# Synthetic

In [9]:
synth_ox= pd.read_csv('data/synth_OX_20.csv')

# split synth into features (x) and target (y)
ox_inputs= synth_ox.loc[:,synth_ox.columns[0:20]]
ox_labels= synth_ox.loc[:,synth_ox.columns[20:21]]

ox_inputs.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
0,0.325761,0.472693,0.385410,0.312884,0.271236,0.207762,0.620768,0.083032,0.380621,0.385330,0.661285,0.607623,0.369915,0.662277,0.266994,0.327500,0.532432,0.394437,0.340003,0.429918
1,0.391322,0.439526,0.399563,0.394650,0.292309,0.425911,0.405543,0.767337,0.396363,0.373539,0.434264,0.481565,0.244531,0.238732,0.337101,0.624010,0.301257,0.388973,0.563497,0.570738
2,0.381488,0.723214,0.272272,0.415180,0.283829,0.428776,0.359133,0.394703,0.170128,0.264208,0.340062,0.436833,0.357293,0.505398,0.311589,0.651714,0.489318,0.498270,0.449700,0.531800
3,0.306399,0.198559,0.360769,0.301099,0.801582,0.207521,0.348990,0.240065,0.317928,0.134672,0.538223,0.613235,0.373322,0.494310,0.436847,0.372760,0.671813,0.506265,0.534691,0.509233
4,0.298658,0.286087,0.301725,0.460478,0.515515,0.264769,0.375526,0.297023,0.271326,0.711050,0.506832,0.484622,0.405023,0.355244,0.330762,0.502291,0.423997,0.393348,0.255827,0.477407


In [10]:
print(np.round(ox_inputs.memory_usage().sum() / 10**6, 2), "MB")

0.16 MB


In [13]:
# split ox_inputs and df_labels into train (80%) and test (20%) datasets

train_ox, test_ox, labels_train_ox, labels_test_ox= sklearn.model_selection.train_test_split(ox_inputs,
                                                                                             ox_labels,
                                                                                             train_size=0.80,
                                                                                             random_state=1234)
train_ox.head()

,ft_1,ft_2,ft_3,ft_4,ft_5,ft_6,ft_7,ft_8,ft_9,ft_10,ft_11,ft_12,ft_13,ft_14,ft_15,ft_16,ft_17,ft_18,ft_19,ft_20
281,0.705423,0.250877,0.393413,0.353529,0.328286,0.374804,0.312217,0.130608,0.281738,0.368790,0.758551,0.608015,0.440911,0.481930,0.263193,0.348485,0.127619,0.420548,0.382225,0.347964
42,0.119699,0.665472,0.235255,0.404961,0.292727,0.321720,0.246230,0.324422,0.360535,0.228708,0.630680,0.377279,0.328190,0.445910,0.344165,0.403461,0.579152,0.558943,0.448307,0.455760
255,0.226558,0.670225,0.344595,0.497451,0.298231,0.316184,0.430817,0.313977,0.414127,0.390934,0.637601,0.566323,0.420372,0.513464,0.609983,0.332080,0.327387,0.239984,0.463658,0.595457
906,0.229718,0.291506,0.394624,0.390758,0.310606,0.680242,0.283559,0.345412,0.421726,0.394265,0.589889,0.494661,0.640523,0.478399,0.437791,0.589921,0.648241,0.700112,0.269150,0.385109
394,0.373117,0.249822,0.259671,0.329655,0.307589,0.657956,0.408162,0.318289,0.333588,0.428981,0.631567,0.439209,0.392151,0.282901,0.562405,0.525431,0.416705,0.713325,0.586582,0.639019
